# NHF validation

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_Te(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(230, 280))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Tw(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(180, 230))
    return x.sel(idx).mean(["longitude", "latitude"])

## load data

### ERA5

#### heat/radiation fluxes

In [ ]:
## nhf
nhf_era5 = xr.open_dataset(DATA_FP / "era5_nhf_glade.nc").drop_vars("utc_date")

## make sure latitude is ascending
nhf_era5 = nhf_era5.reindex({"latitude": nhf_era5.latitude[::-1]})

## rename flux variables
varname_dict = dict(
    MSLHF="lhf",
    MSSHF="shf",
    MSNLWRF="lw",
    MSNLWRFCS="lw_cs",
    MSNSWRF="sw",
    MSNSWRFCS="sw_cs",
)
nhf_era5 = nhf_era5.rename(varname_dict)

## net heat flux
## Note: "The ECMWF convention for vertical fluxes is positive downwards."
nhf_era5["nhf"] = nhf_era5["sw"] + nhf_era5["lw"] + nhf_era5["lhf"] + nhf_era5["shf"]

#### SST

In [ ]:
## load ERSST
REF_FP = pathlib.Path("/glade/campaign/cgd/cas/asphilli/CVDP-OBS/")
# pr_ref = xr.open_dataset(REF_FP / "gpcp.mon.mean.197901-202403.nc")["precip"]
sst_ersst = xr.merge(
    [
        xr.open_dataset(REF_FP / "ersstv5.185401-202412.nc")["ssta"],
        xr.open_dataset(REF_FP / "ersstv5.185401-202412.nc")["sst"],
        # xr.open_dataset(REF_FP / "gpcp.mon.mean.197901-202403.nc")["precip"],
    ]
)

sst_ersst = sst_ersst.rename({"lon": "longitude", "lat": "latitude"})
sst_ersst["time"] = xr.date_range(start="1854-01", end="2024-12", freq="MS")
sst_ersst = sst_ersst.sel(time=nhf_era5.time)

### CESM2

In [ ]:
## load data
_, anom = src.utils.load_consolidated()

## load flux data
_, flux_anom = src.utils.load_flux_data()

## update sign of LHFLX
flux_anom["lhflx"] = -flux_anom["lhflx"]

## merge
anom = xr.merge([anom, flux_anom])

## subset in time
anom = anom.sel(time=slice("1950", "1980"))

## rename flux variables
varname_dict_cesm = dict(
    sst="ssta",
    lhflx="lhf",
    flns="lw",
    fsns="sw",
    fsnsc="sw_cs",
)

## add component information
for v0 in list(varname_dict_cesm.keys()):
    varname_dict_cesm[f"{v0}_comp"] = f"{varname_dict_cesm[v0]}_comp"

## update names
anom = anom.rename(varname_dict_cesm)

## subset for given variables
varnames = ["ssta", "pr", "nhf", "sw", "lw", "lhf", "sw_cs"]
varnames += [f"{v}_comp" for v in varnames]
anom = anom[varnames]

### Compute indices

In [ ]:
fns = dict(
    _3=src.utils.get_nino3,
    _34=src.utils.get_nino34,
    _4=src.utils.get_nino4,
    _e=get_Te,
    _w=get_Tw,
)

#### ERA5

In [ ]:
idx_obs = xr.Dataset()
# idx_mod = xr.Dataset()
for ds in [nhf_era5, sst_ersst[["ssta"]]]:
    for fn_name, fn in fns.items():
        for v in list(ds):
            idx_obs[f"{v}{fn_name}"] = fn(ds[v])

#### CESM2

In [ ]:
idx_mod = xr.Dataset()

for n, fn in tqdm.tqdm(fns.items()):

    ## compute
    idx_mod_ = src.utils.reconstruct_wrapper(anom, fn)

    ## rename
    name_dict = dict(zip(list(idx_mod_), [f"{i}{n}" for i in list(idx_mod_)]))
    idx_mod = xr.merge([idx_mod, idx_mod_.rename(name_dict)])

### resample to NDJ

In [ ]:
def resample_ndj(x):
    """resample data to NDJ avg"""

    ## resample and trim
    x = x.resample({"time": "QS-NOV"}).mean().isel(time=slice(4, -4, 4))

    ## remove mean
    return x - x.mean()


idx_obs = resample_ndj(idx_obs)
idx_mod = resample_ndj(idx_mod)

normalized version


In [ ]:
for v in list(idx_obs):
    idx_obs[f"{v}_norm"] = idx_obs[v] / idx_obs[v].std()

for v in list(idx_mod):
    idx_mod[f"{v}_norm"] = idx_mod[v] / idx_mod[v].std()

## Plots

### ERA5

#### Scatter and timeseries

In [ ]:
## useful things
yr = idx_obs.time.dt.year
ax_kwargs = dict(c="k", ls="--", lw=0.8, zorder=0.5)

for suf in ["e", "w"]:
    print(f"\nLoc: {suf}")

    fig, axs = plt.subplots(2, 5, figsize=(10, 4.5), layout="constrained")

    for j, v in enumerate(["nhf", "sw", "lhf", "lw", "shf"]):

        ## scatter plot
        axs[0, j].scatter(
            idx_obs[f"ssta_{suf}"],
            idx_obs[f"{v}_{suf}"],
            s=20,
        )
        axs[0, j].set_title(v)
        axs[0, j].axvline(0, **ax_kwargs)
        axs[0, j].axhline(0, **ax_kwargs)
        axs[0, j].set_xticks([-2, 0, 2])
        axs[0, j].set_xlabel("ssta")

        ## timeseries
        axs[1, j].plot(yr, idx_obs[f"{v}_{suf}"])
        axs[1, j].axhline(0, **ax_kwargs)
        axs[1, j].axvline(1979, **ax_kwargs)
        axs[1, j].set_xticks([1979, 2022])

    src.utils.set_lims(axs[0])
    src.utils.set_lims(axs[1])
    for ax in axs[:, 0]:
        ax.set_ylabel(r"W m$^{-2}$")
    for ax in axs[:, 1:].flatten():
        ax.set_yticks([])
    plt.show()

#### SST over time

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 2.25), layout="constrained")

## timeseries
for ax, suf in zip(axs[:2], ["w", "e"]):
    ax.plot(yr, idx_obs[f"ssta_{suf}"])
    ax.axhline(0, **ax_kwargs)
    ax.axvline(1979, **ax_kwargs)
    ax.set_xticks([1979, 2022])

## scatter relationship
axs[-1].scatter(idx_obs[f"ssta_e"], idx_obs[f"ssta_w"], s=15)
axs[-1].axhline(0, **ax_kwargs)
axs[-1].axvline(0, **ax_kwargs)
axs[-1].set_aspect("equal")
zz = np.linspace(-2, 2.5)
# axs[-1].plot(zz, zz, **dict(ax_kwargs, zorder=10))


## format/labels
src.utils.set_lims(axs[:2])
axs[1].set_yticks([])
axs[0].set_title(r"central SST")
axs[1].set_title(r"east SST")
axs[2].set_ylabel(r"central")
axs[2].set_xlabel(r"east")

plt.show()

#### Comparison between ERA5 and CESM2

##### funcs to get best fit

In [ ]:
def psi_poly(T, c, b, a, e):
    """base transfer function"""
    # return e * x**3 + c * x**2 + b * x + a
    return (c / 3) * T**3 + (b / 2) * T**2 + a * T + e


def get_psi_poly(x, y):
    """get transfer function from data"""

    ## prep data
    if "member" in x.dims:
        stack = lambda x: x.stack(s=["time", "member"]).values
        x = stack(x)
        y = stack(y)

    ## get parameter fit
    p, _ = scipy.optimize.curve_fit(
        f=psi_poly,
        xdata=x,
        ydata=y,
    )

    ## define transfer func
    psi = lambda T: psi_poly(T, c=p[0], b=p[1], a=p[2], e=p[3])

    ## put params in dataarray
    params = xr.DataArray(p, coords=dict(param=["c", "b", "a", "e"])).rename("params")

    return psi, params


def get_psi_wrapper(xy, x_var, y_var, get_psi_func=get_psi_poly):
    """get transfer function from data"""

    ## get transfer func
    psi_func, p = get_psi_func(x=xy[x_var], y=xy[y_var])

    ## get coefficient array (standardized units)
    z = np.arange(-3.5, 3.6, 0.1)
    z = xr.DataArray(z, coords=dict(sigma=z))

    ## evaluate function
    psi_eval = xr.zeros_like(z)
    psi_eval.values = psi_func(z.values)

    ## merge eval and params
    return xr.merge([psi_eval.rename("psi_eval"), p])

##### Make plot

In [ ]:
## useful things
yr = idx_obs.time.dt.year
ax_kwargs = dict(c="k", ls="--", lw=0.8, zorder=0.5)

for suf in ["e", "w"]:

    ## print out location
    print(f"\nLoc: {suf}")

    ## empty array to hold psi funcs
    psi_evals = np.empty(shape=(2, 4), dtype=object)

    fig, axs = plt.subplots(2, 4, figsize=(8, 4.5), layout="constrained")

    ## loop thru flux components
    for j, v in enumerate(["nhf", "sw", "lhf", "lw"]):

        ## label
        axs[0, j].set_title(v)
        axs[1, j].set_xlabel("ssta")

        ## get vars
        xvar, yvar = f"ssta_{suf}_norm", f"{v}_{suf}"

        ## loop thru model/obs
        for i, (idx_, s, ls, a) in enumerate(
            zip([idx_obs, idx_mod], [20, 1], ["-", "--"], [1, 0.5])
        ):

            ## scatter plot
            axs[i, j].scatter(idx_[xvar], idx_[yvar], s=s, alpha=a)

            ## get and plot best fit
            psi_eval = get_psi_wrapper(idx_, x_var=xvar, y_var=yvar)["psi_eval"]
            min_ = idx_[xvar].values.min()
            max_ = idx_[xvar].values.max()
            psi_evals[i, j] = psi_eval.sel(sigma=slice(min_, max_))
            axs[i, j].plot(psi_evals[i, j].sigma, psi_evals[i, j], c="k", ls=ls)

            ## guidelines
            axs[i, j].axvline(0, **ax_kwargs)
            axs[i, j].axhline(0, **ax_kwargs)
            axs[i, j].set_xticks([-2, 0, 2])

    ## plot obs. fits on model
    for j, ax in enumerate(axs[1]):
        ax.plot(psi_evals[0, j].sigma, psi_evals[0, j], c="k", ls="-")

    ## formatting
    src.utils.set_lims(axs)
    for ax in axs[:, 0]:
        ax.set_ylabel(r"W m$^{-2}$")
    for ax in axs[:, 1:].flatten():
        ax.set_yticks([])

    for ax in axs[0, :]:
        ax.set_xticks([])
    plt.show()

### 

## to-do
- same plots, but for CESM2
- relative SST comparison between ERSST and CESM2 (spatial plot?)
- rainfall comparison?